In [0]:
dbutils.widgets.text("catalog", "dev")
CURRENT_CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CURRENT_CATALOG}.kruger")
print(f"Verified schema: {CURRENT_CATALOG}.kruger is ready.")


In [0]:
# Databricks notebook source
%pip install google-api-python-client

# COMMAND ----------

import os
import json
from datetime import datetime
from googleapiclient.discovery import build

# Setup widgets to accept the incoming variables from the asset bundle task parameters
dbutils.widgets.text("catalog", "dev") 
dbutils.widgets.text("youtube_api_key", "")
dbutils.widgets.text("youtube_channel_id", "")

CURRENT_CATALOG = dbutils.widgets.get("catalog")
API_KEY = dbutils.widgets.get("youtube_api_key")
CHANNEL_ID = dbutils.widgets.get("youtube_channel_id")

# Fail early if deployment parameters weren't correctly provided via GitHub / Bundle deployment
if not API_KEY or not CHANNEL_ID:
    raise ValueError("Missing required runtime configurations! Please verify 'youtube_api_key' and 'youtube_channel_id' parameters are populated.")

print(f"Running pipeline using Unity Catalog: {CURRENT_CATALOG}")

# Generate the correct uploads playlist ID dynamically from the Channel ID
UPLOADS_PLAYLIST_ID = "UU" + CHANNEL_ID[2:]  

VOLUME_DIR = f"/Volumes/{CURRENT_CATALOG}/kruger/landing/raw_youtube/"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
VOLUME_PATH = f"{VOLUME_DIR}latest_sightings_{timestamp}.json"

# Build the client service object securely
youtube = build("youtube", "v3", developerKey=API_KEY)

try:
    print(f"Checking for recent videos from Latest Sightings channel ({CHANNEL_ID})...")
    
    playlist_req = youtube.playlistItems().list(
        part="contentDetails",
        playlistId=UPLOADS_PLAYLIST_ID,
        maxResults=50
    )
    playlist_resp = playlist_req.execute()
    video_ids = [item['contentDetails']['videoId'] for item in playlist_resp.get('items', [])]
    
    if video_ids:
        stats_req = youtube.videos().list(
            part="snippet,statistics",
            id=",".join(video_ids)
        )
        stats_resp = stats_req.execute()
        
        os.makedirs(VOLUME_DIR, exist_ok=True)
        with open(VOLUME_PATH, "w", encoding="utf-8") as json_file:
            json.dump(stats_resp, json_file, indent=4)
            
        print(f" Successfully landed raw file in Volume: {VOLUME_PATH}")
    else:
        print("No recent videos found.")

except Exception as e:
    print(f"Pipeline Error: {e}")
